# 🌍 ICARUS_WORKER_SOLID_EARTH — Solid Earth & Seismology Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 3)

**Role:** Solid Earth, Geophysics & Seismic Waveform Analysis  
**Primary Model:** SeisBench PhaseNet (Zhu & Beroza, 2019)  
**Pretrained Checkpoint:** `stead` (Stanford Earthquake Dataset)  
**Input Format:** 3-Component Seismic Waveforms `[Z, N, E]` at 100 Hz  
**Deployment Profile:** Kaggle GPU (T4 or P100) or CPU

---

### Worker Responsibilities
- Hosts deterministic seismology algorithms: **tapering, detrending, SNR computation, FFT spectrum, Arias intensity**.
- Hosts **SeisBench PhaseNet** for automatic $P$-wave and $S$-wave arrival picking.
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


In [ ]:
# 1. Environment & Dependencies Setup
import os
import sys
from pathlib import Path

!pip install -q seisbench obspy pyngrok uvicorn fastapi pydantic

# Clone or update ICARUS repository
!git clone https://github.com/kanishk10-ux/ICARUS.git /kaggle/working/ICARUS 2>/dev/null || (cd /kaggle/working/ICARUS && git pull)
sys.path.insert(0, "/kaggle/working/ICARUS")

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. PhaseNet Adapter Verification & Smoke Test
import numpy as np
from icarus.workers.solid_earth.phasenet_adapter import PhaseNetAdapter
from icarus.algorithms.solid_earth import SolidEarthAlgorithms

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = PhaseNetAdapter(pretrained_weights="stead", device=device)

ok, msg = adapter.load_model()
print("Load Status:", ok, "->", msg)

# Create realistic synthetic 3-component seismic waveform (30 seconds at 100 Hz = 3000 samples)
np.random.seed(42)
t = np.linspace(0, 30, 3000)
noise = np.random.normal(0, 0.05, (3, 3000))
# Add synthetic P-wave arrival at t=10s and S-wave arrival at t=18s
p_signal = np.sin(2 * np.pi * 8 * (t - 10)) * np.exp(-0.5 * (t - 10)) * (t >= 10)
s_signal = np.sin(2 * np.pi * 4 * (t - 18)) * np.exp(-0.3 * (t - 18)) * (t >= 18)
waveform_3c = noise
waveform_3c[0] += p_signal * 0.8  # Z-component (P dominant)
waveform_3c[1] += s_signal * 1.0  # N-component (S dominant)
waveform_3c[2] += s_signal * 0.9  # E-component

picks_result = adapter.predict_phases(waveform_3c, sampling_rate=100.0, threshold=0.3)
print("PhaseNet Result Status:", picks_result["status"])
print(f"Detected {picks_result['total_p_picks']} P-picks and {picks_result['total_s_picks']} S-picks")
for p in picks_result["p_picks"]:
    print(f"  P-wave Pick: t={p['time_offset_seconds']}s, Confidence={p['confidence']}")
for s in picks_result["s_picks"]:
    print(f"  S-wave Pick: t={s['time_offset_seconds']}s, Confidence={s['confidence']}")

# Waveform metrics via scientific algorithms layer
snr = SolidEarthAlgorithms.compute_snr_db(waveform_3c[0], noise_window_samples=800, signal_window_samples=800)
spec = SolidEarthAlgorithms.compute_frequency_spectrum(waveform_3c[0], sampling_rate_hz=100.0)
print(f"Waveform SNR: {snr} dB | Dominant Frequency: {spec['dominant_frequency_hz']} Hz")

In [ ]:
# 3. Launch Dedicated Worker Service with Public ngrok Tunnel
from scripts.worker_bootstrap import create_worker_app
import threading
import uvicorn

app = create_worker_app("solid_earth", device=device)

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("✅ Solid Earth Worker running locally on port 8001")

# Connect ngrok if secret configured
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8001, "http")
    print("\n🌐 SOLID EARTH PUBLIC WORKER TUNNEL:", tunnel.public_url)
    print("👉 Register this endpoint with ICARUS_CORE.")
except Exception as e:
    print("Tunnel notice:", e)